In [5]:
%pip install pandas
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
import pandas as pd

# ==========================================
# 1. โหลดข้อมูล (ใส่ engine='calamine' เพื่อเลี่ยง XML เสียหายจากรอบแรก)
# ==========================================
file_path = r"C:\Users\KS\Desktop\product.xlsx"
df = pd.read_excel(file_path, engine="calamine")

# ล้างช่องว่างที่อาจมองไม่เห็นในชื่อคอลัมน์ทิ้งให้หมดเพื่อความปลอดภัย
df.columns = df.columns.str.strip()

df = df[df['import'] > 0]
# ==========================================
# 2. คำนวณหา Outlier แบบประวัติตัวมันเอง (วิธีที่เสถียรที่สุด)
# ==========================================
# ใช้ .transform() เพื่อหาค่า Mean และ SD แยกกลุ่มสินค้า แต่รักษาโครงสร้างตารางเดิมไว้ร้อยเปอร์เซ็นต์
group_mean = df.groupby("product_id")["import"].transform("mean")
group_mad = df.groupby("product_id")["import"].transform("std")

# คำนวณ Z-score โดยระวังกรณีที่ค่า SD เป็น 0 (ยอดขายนิ่งสนิท)
# ถ้า SD เป็น 0 หรือหาค่าไม่ได้ ให้ Z-score เป็น 0
df["z_score"] = np.where(
    (group_mad == 0) | (group_mad.isna()),  # [IF]   ถ้า SD เป็น 0 หรือหาค่าไม่ได้
    0,  # [THEN] ให้ Z-score เป็น 0 ไปเลย (เพื่อไม่ให้สูตรคณิตศาสตร์พัง)
    (df["import"] - group_mean) / group_mad,  # [ELSE] ถ้าปกติ ก็จับลบกันแล้วหารด้วย SD ตามสูตรปกติ
)

# กรองเกณฑ์ที่เริ่มแกว่ง (ปรับตัวเลขจาก 2 เป็น 1.5 ได้ตามความไวที่ต้องการ)
df["is_outlier"] = df["z_score"].abs() > 3


# ==========================================
# 3. เจาะลึกระดับบิล (Drill-Down)
# ==========================================
# ดึงรายชื่อรหัสสินค้าทั้งหมดที่มีแถวใดแถวหนึ่งติดสถานะ Outlier
outlier_products = df[df["is_outlier"] == True]["product_id"].unique()

# ดึง "ทุกบิล" ของสินค้าในกลุ่ม outlier_products ขึ้นมาดูประวัติเปรียบเทียบ
final_report = df[df["product_id"].isin(outlier_products)].copy()

# จัดเรียงข้อมูลให้ดูง่าย: เรียงตามรหัสสินค้า และเอาบิลที่แกว่งที่สุด (Z-score สูงสุด) ขึ้นก่อน
final_report = final_report.sort_values(
    by=["product_id", "z_score"], ascending=[True, False]
)


In [26]:
excel = final_report[["DATE", "product_id", "Bill", "import", "z_score"]]

 Robust Z-score

In [ ]:
%pip install scipy

In [3]:
%pip install calamine

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement calamine (from versions: none)

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for calamine


In [27]:
import numpy as np
import pandas as pd
from scipy.stats import median_abs_deviation
import openpyxl
# ==========================================
# 1. โหลดข้อมูล (ใส่ engine='calamine' เพื่อเลี่ยง XML เสียหายจากรอบแรก)
# ==========================================
file_path = r"C:\Users\KS\Desktop\สต็อกรวม.xlsx"
df = pd.read_excel(file_path, engine="calamine")

# ล้างช่องว่างที่อาจมองไม่เห็นในชื่อคอลัมน์ทิ้งให้หมดเพื่อความปลอดภัย
df.columns = df.columns.str.strip()

# [เสริมเกราะ 1] แปลง ID ให้เป็น string ทั้งหมด ป้องกันกรณี Excel แปลงบางตัวเป็นตัวเลขแล้วกลุ่มเพี้ยน
df['product_id'] = df['product_id'].astype(str).str.strip()

# เอาเฉพาะยอดนำเข้าที่มากกว่า 0 เท่านั้น (ตัด Noise/บิลยกเลิก ออก)
df = df[df['import'] > 0]
# ==========================================
# 2. คำนวณหา Outlier แบบประวัติตัวมันเอง (วิธีที่เสถียรที่สุด)
# ==========================================
# ใช้ .transform() เพื่อหาค่า Mean และ SD แยกกลุ่มสินค้า แต่รักษาโครงสร้างตารางเดิมไว้ร้อยเปอร์เซ็นต์
group_median = df.groupby("product_id")["import"].transform("median")
group_mad =  df.groupby('product_id')['import'].transform(
    lambda x: median_abs_deviation(x, scale='normal')
)

# คำนวณ Z-score โดยระวังกรณีที่ค่า SD เป็น 0 (ยอดขายนิ่งสนิท)
# ถ้า SD เป็น 0 หรือหาค่าไม่ได้ ให้ Z-score เป็น 0
df["Robustz_score"] = np.where(
    (group_mad == 0) | (group_mad.isna()),  # [IF]   ถ้า SD เป็น 0 หรือหาค่าไม่ได้
    0,  # [THEN] ให้ Z-score เป็น 0 ไปเลย (เพื่อไม่ให้สูตรคณิตศาสตร์พัง)
    (df["import"] - group_median) / group_mad  # [ELSE] ถ้าปกติ ก็จับลบกันแล้วหารด้วย SD ตามสูตรปกติ
)

# กรองเกณฑ์ที่เริ่มแกว่ง (ปรับตัวเลขจาก 2 เป็น 1.5 ได้ตามความไวที่ต้องการ)
df["is_outlier"] = df["Robustz_score"].abs() > 3


# ==========================================
# 3. เจาะลึกระดับบิล (Drill-Down)
# ==========================================
# ดึงรายชื่อรหัสสินค้าทั้งหมดที่มีแถวใดแถวหนึ่งติดสถานะ Outlier
final_report = df[df["is_outlier"] == True].copy()

# จัดเรียงข้อมูลให้ดูง่ายเหมือนเดิม
final_report = final_report.sort_values(
    by=["product_id", "Robustz_score"], ascending=[True, False]
)

In [29]:
final_report.to_excel(r'C:\Users\KS\Desktop\product_out1.xlsx',index=False,engine='openpyxl')

In [28]:
final_report

,DATE,Bill,details,product_id,import,export,balance,Unnamed: 7,Robustz_score,is_outlier
1933,05/04/2569,IBK3256904/017,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,น้ำ เพียวไลฟ์ 600มล.,300.0,0.0,330.07,NaN,18.885713,True
1961,17/04/2569,IBK3256904/053,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,น้ำ เพียวไลฟ์ 600มล.,300.0,0.0,301.09,NaN,18.885713,True
1997,04/05/2569,IBK3256905/010,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,น้ำ เพียวไลฟ์ 600มล.,300.0,0.0,393.00,NaN,18.885713,True
2042,25/05/2569,IBK3256905/080,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,น้ำ เพียวไลฟ์ 600มล.,300.0,0.0,402.10,NaN,18.885713,True
2089,15/06/2569,IBK3256906/041,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,น้ำ เพียวไลฟ์ 600มล.,300.0,0.0,364.05,NaN,18.885713,True
...,...,...,...,...,...,...,...,...,...,...
794,12/01/2569,IBK3256901/037,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,ไวไวซอง หมูสับ,20.0,0.0,34.00,NaN,3.597279,True
822,05/02/2569,IBK3256902/014,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,ไวไวซอง หมูสับ,20.0,0.0,31.00,NaN,3.597279,True
843,03/03/2569,IBK3256903/004,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,ไวไวซอง หมูสับ,20.0,0.0,35.00,NaN,3.597279,True
878,16/04/2569,IBK3256904/048,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,ไวไวซอง หมูสับ,20.0,0.0,22.00,NaN,3.597279,True
